# LinkedIn Title Lookup
Searches Google **and** Bing for each contact's LinkedIn job title.  
Progress is saved after every row — safe to interrupt and resume.

In [ ]:
# !pip install requests beautifulsoup4 pandas ipywidgets

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import time, random, json, os, re
import ipywidgets as widgets
from IPython.display import display, FileLink

CHECKPOINT_FILE = 'checkpoint.json'
OUTPUT_FILE     = 'output_with_titles.csv'

upload_btn = widgets.FileUpload(accept='.csv', multiple=False)
display(upload_btn)

In [ ]:
def read_uploaded_csv(w):
    import io
    content = list(w.value.values())[0]['content']
    return pd.read_csv(io.BytesIO(content))

data = read_uploaded_csv(upload_btn)
print(f'Loaded {len(data)} rows')
data.head(3)

In [ ]:
# ── HTTP helpers ────────────────────────────────────────────────────────────

UA_LIST = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15',
    'Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0',
]

def make_session():
    s = requests.Session()
    s.headers.update({
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    })
    return s

def is_captcha_page(html: str) -> bool:
    markers = ['detected unusual traffic', 'captcha', 'recaptcha', '/sorry/index']
    low = html.lower()
    return any(m in low for m in markers)

def safe_get(session, url, retries=3):
    """GET with random UA, retries on 5xx, returns (response|None, blocked:bool)."""
    for attempt in range(retries):
        session.headers['User-Agent'] = random.choice(UA_LIST)
        try:
            resp = session.get(url, timeout=14)
            if is_captcha_page(resp.text):
                return None, True
            if resp.status_code == 429:
                return None, True
            if resp.status_code >= 500:
                time.sleep(4 * (attempt + 1))
                continue
            return resp, False
        except Exception:
            time.sleep(4 * (attempt + 1))
    return None, False


# ── Title extraction ─────────────────────────────────────────────────────────

def _extract_title(text: str) -> str:
    """Pull job title from a snippet like 'Title · Company · ...' or 'Title at Company'."""
    for junk in ['Web result with Site Links', 'More results', '\n']:
        text = text.replace(junk, ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    if '·' in text:
        candidate = text.split('·')[0].strip()
        if 3 < len(candidate) < 120 and 'linkedin' not in candidate.lower():
            return candidate

    m = re.match(r'^(.{3,80}?)\s+at\s+', text, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    return ''


def _parse_snippets(soup: BeautifulSoup) -> str:
    """Walk all text blocks in a search-result page and return the first plausible title."""
    for el in soup.find_all(['span', 'div', 'p']):
        # Skip huge containers that mix many results
        if el.find(['div', 'article']):
            continue
        text = el.get_text(separator=' ').strip()
        if 20 < len(text) < 350 and ('·' in text or re.search(r'\bat\b', text, re.I)):
            title = _extract_title(text)
            if title:
                return title
    return ''


# ── Per-engine search ─────────────────────────────────────────────────────────

def search_google(full_name, company, session):
    q = f'site:linkedin.com/in "{full_name}" "{company}"'
    url = 'https://www.google.com/search?q=' + quote_plus(q) + '&hl=en&num=5'
    resp, blocked = safe_get(session, url)
    if blocked:
        return '', True
    if resp is None:
        return '', False
    return _parse_snippets(BeautifulSoup(resp.text, 'html.parser')), False


def search_bing(full_name, company, session):
    q = f'site:linkedin.com/in "{full_name}" "{company}"'
    url = 'https://www.bing.com/search?q=' + quote_plus(q) + '&count=5'
    resp, blocked = safe_get(session, url)
    if blocked:
        return '', True
    if resp is None:
        return '', False
    return _parse_snippets(BeautifulSoup(resp.text, 'html.parser')), False


def get_title(full_name, company, google_sess, bing_sess):
    """
    Try Google first; fall back to Bing.
    Returns (title, google_blocked, bing_blocked).
    """
    title, g_blocked = search_google(full_name, company, google_sess)
    if title:
        return title, g_blocked, False

    title, b_blocked = search_bing(full_name, company, bing_sess)
    return title or '[not found]', g_blocked, b_blocked


print('Functions ready.')

In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)   # {str(index): title}
    return {}

def save_checkpoint(results: dict):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(results, f)


# ── Main loop ─────────────────────────────────────────────────────────────────

df = data.copy()
df['Full Name'] = (df['First Name'].fillna('') + ' ' + df['Last Name'].fillna('')).str.strip()
df['LinkedIn Job Title'] = ''

results     = load_checkpoint()
google_sess = make_session()
bing_sess   = make_session()

google_blocked_count = 0
GOOGLE_BLOCK_PAUSE   = 120   # seconds to wait when Google blocks us

for i, row in df.iterrows():
    key = str(i)
    if key in results:           # already done — skip
        df.at[i, 'LinkedIn Job Title'] = results[key]
        print(f'[{i+1}/{len(df)}] (cached) {row["Full Name"]} → {results[key]}')
        continue

    full_name = row['Full Name']
    company   = str(row.get('Outlook Company', '')).strip()

    print(f'[{i+1}/{len(df)}] {full_name} @ {company} ...', end=' ', flush=True)

    title, g_blocked, b_blocked = get_title(full_name, company, google_sess, bing_sess)

    if g_blocked:
        google_blocked_count += 1
        print(f'⚠️  Google blocked (count={google_blocked_count}). Pausing {GOOGLE_BLOCK_PAUSE}s ...', flush=True)
        # Replace the session so we get a fresh cookie jar
        google_sess = make_session()
        time.sleep(GOOGLE_BLOCK_PAUSE)
        # Retry once after the pause
        title, _, b_blocked2 = get_title(full_name, company, google_sess, bing_sess)

    df.at[i, 'LinkedIn Job Title'] = title
    results[key] = title
    save_checkpoint(results)

    # Save partial CSV so nothing is lost
    df.to_csv(OUTPUT_FILE, index=False)

    print(title)

    # Delay: 8-15 s normally; add extra time every 20 rows to cool down
    base_delay = random.uniform(8, 15)
    if (i + 1) % 20 == 0:
        base_delay += random.uniform(30, 60)
        print(f'  → 20-row cooldown pause ({base_delay:.0f}s) ...')
    time.sleep(base_delay)

print('\nDone!')
df[['Full Name', 'Outlook Company', 'Primary Email', 'LinkedIn Job Title']].head(10)

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(df)} rows to {OUTPUT_FILE}')
FileLink(OUTPUT_FILE)